# Data collection

In [151]:
import os
import requests
import time
import pandas as pd
from dotenv import load_dotenv
import time
import glob

# Load environment variables from the .env file
load_dotenv()

# Grab the key securely
API_KEY = os.getenv("RIOT_API_KEY")

if not API_KEY:
    print("WARNING: API Key not found. Check your .env file!")

headers = {
    "X-Riot-Token": API_KEY
}

# Riot API uses two different routing systems depending on the endpoint.
# Platform routing (na1, euw1) is used for ladders and summoner profiles.
# Regional routing (americas, europe) is used for match histories.
PLATFORM = "euW1" 
REGION = "europe"

## Ratelimit helper

In [152]:
def make_api_request(url):

    #A wrapper for requests.get() that automatically handles standard Riot API rate limits.
    while True:
        response = requests.get(url, headers=headers)
        
        if response.status_code == 200:
            return response.json()
            
        elif response.status_code == 429: # Too Many Requests 
            retry_after = int(response.headers.get("Retry-After", 10)) 
            print(f"Rate limit hit. Sleeping for {retry_after} seconds...")
            time.sleep(retry_after)
            
        elif response.status_code in [403, 401]:
            print("API Key expired or invalid. Please check your Riot Developer portal.")
            print("status_code:", response.status_code)
            return None
            
        else:
            print(f"Error {response.status_code} for URL: {url}")
            return None

## Get match history and parse the data

In [153]:
def get_player_match_history(puuid, total_matches=250):
    """
    Fetches a large number of Match IDs for a player by paginating 
    through the API 100 matches at a time.
    """
    all_match_ids = []
    start_index = 0
    
    print(f"Fetching up to {total_matches} match IDs for player...")
    
    while len(all_match_ids) < total_matches:
        # Calculate how many matches to ask for in this specific request
        # (It will be 100, unless we only need a few more to hit our total)
        matches_left = total_matches - len(all_match_ids)
        current_count = min(100, matches_left)
        
        # queue=420 filters for Ranked Solo/Duo
        url = f"https://{REGION}.api.riotgames.com/lol/match/v5/matches/by-puuid/{puuid}/ids?queue=420&start={start_index}&count={current_count}"
        
        print(f" -> Requesting {current_count} matches starting at index {start_index}...")
        batch_ids = make_api_request(url)
        
        # If the API returns an empty list, we've reached the end of their match history!
        if not batch_ids:
            print("Reached the end of the player's match history.")
            break
            
        all_match_ids.extend(batch_ids)
        start_index += current_count
        
        # Respect the rate limit between pagination requests
        time.sleep(1.2)
        
    print(f"Successfully grabbed {len(all_match_ids)} total match IDs.")
    return all_match_ids

In [154]:
def parse_match_data(match_id, target_puuid):
    """
    Pulls the detailed stats for a match and extracts the specific 
    variables needed for session analysis.
    """
    url = f"https://{REGION}.api.riotgames.com/lol/match/v5/matches/{match_id}"
    match_data = make_api_request(url)
    
    if not match_data or 'info' not in match_data:
        return None

    info = match_data['info']
    
    # Find our target player among the 10 participants
    participants = info.get('participants', [])
    player_data = next((p for p in participants if p['puuid'] == target_puuid), None)
    
    if not player_data:
        return None

    # Extract the core data points for your project
    parsed_data = {
        "match_id": match_id,
        "puuid": target_puuid,
        "game_start_timestamp": info.get("gameStartTimestamp"), # Milliseconds since epoch
        "game_end_timestamp": info.get("gameEndTimestamp"),     # Milliseconds since epoch
        "game_duration_sec": info.get("gameDuration"),          # Seconds
        "win": player_data.get("win"),                          # Boolean (True/False)
        "champion_played": player_data.get("championName"),
        "kills": player_data.get("kills"),
        "deaths": player_data.get("deaths"),
        "assists": player_data.get("assists")
    }
    
    return parsed_data

## Player Rank

In [155]:
def get_player_rank(puuid):
    """
    Fetches a player's Ranked Solo/Duo rank directly using their PUUID.
    Returns a string like 'GOLD II' or 'UNRANKED'.
    """
    league_url = f"https://{PLATFORM}.api.riotgames.com/lol/league/v4/entries/by-puuid/{puuid}"
    league_data = make_api_request(league_url)
    
    if league_data is None:
        return "API_ERROR"
        
    if not league_data: # If the list is empty, they haven't played ranked
        return "UNRANKED"
        
    # A player can have multiple ranks (Solo/Duo, Flex). We only want Solo/Duo.
    for queue in league_data:
        if queue.get('queueType') == 'RANKED_SOLO_5x5':
            tier = queue.get('tier', '')
            rank = queue.get('rank', '')
            return f"{tier} {rank}" # e.g., "DIAMOND IV"
            
    return "UNRANKED (Solo/Duo)"

## Save data

In [156]:
def save_player_data(player_df, puuid):
    """
    Saves a single player's DataFrame to the Data/match/ folder.
    Using the PUUID in the filename ensures we don't overwrite files.
    """
    # Ensure the directory exists
    os.makedirs("Data/match/", exist_ok=True)
    
    if player_df is not None and not player_df.empty:
        filename = f"player_{puuid}_matches.csv"
        file_path = os.path.join("Data/match/", filename)
        
        # Save to CSV without the index column
        player_df.to_csv(file_path, index=False)
        print(f"Saved {len(player_df)} matches to {file_path}")
    else:
        print(f"No data to save for player {puuid[:15]}")

MASTER_PLAYER_FILE = "Data/player_master_list.csv"

def save_player_metadata(puuid, rank):
    """
    Appends a player's PUUID and rank to the master CSV file.
    """
    os.makedirs("Data", exist_ok=True) # Just need the base Data folder now
    
    info_df = pd.DataFrame([{"puuid": puuid, "rank": rank}])
    
    # If the file doesn't exist yet, write it normally (with headers).
    # If it does exist, append to the bottom (mode='a') and skip headers.
    file_exists = os.path.exists(MASTER_PLAYER_FILE)
    info_df.to_csv(MASTER_PLAYER_FILE, mode='a', header=not file_exists, index=False)

## Challenger ladder

In [157]:
def gather_challenger_players(limit=10):
    """
    Pulls top players, grabs their rank, and saves them to the Data/player/ folder.
    """
    print("Fetching Challenger Ladder...")
    ladder_url = f"https://{PLATFORM}.api.riotgames.com/lol/league/v4/challengerleagues/by-queue/RANKED_SOLO_5x5"
    ladder_data = make_api_request(ladder_url)
    
    if not ladder_data:
        return []

    entries = ladder_data.get('entries', [])[:limit]
    saved_count = 0
    
    print(f"Checking and saving PUUIDs for the top {limit} players...")
    
    for entry in entries:
        puuid = entry.get('puuid')
        
        # Fallback for legacy ladder data
        if not puuid and 'summonerId' in entry:
            summoner_id = entry['summonerId']
            sum_url = f"https://{PLATFORM}.api.riotgames.com/lol/summoner/v4/summoners/{summoner_id}"
            sum_data = make_api_request(sum_url)
            if sum_data and 'puuid' in sum_data:
                puuid = sum_data['puuid']
            time.sleep(1.2)
            
        if puuid:
            # Check if we already have this player saved!
            expected_filepath = os.path.join("Data/player/", f"player_{puuid}_info.csv")
            
            if not os.path.exists(expected_filepath):
                print(f"New player found! Fetching rank for {puuid[:15]}...")
                rank = get_player_rank(puuid)
                save_player_metadata(puuid, rank)
                saved_count += 1
            else:
                print(f"Player {puuid[:15]} already in database. Skipping.")
                
    print(f"Ladder discovery complete. Added {saved_count} new players to database.")

## Snowball Method

In [158]:
import time
import os
import pandas as pd

def snowball_continuous(seed_identifier, target_new_players=50, matches_per_player=3):
    """
    Accepts a PUUID, formatted Match ID, or raw Game ID.
    Continuously hops from player to player until the target limit is reached.
    """
    puuids_to_explore = []
    explored_puuids = set()
    saved_count = 0
    
    # ==========================================
    # 1. THE SMART INPUT HANDLER (Queue Seeding)
    # ==========================================
    if "_" in seed_identifier:
        print(f"Seed recognized as a formatted Match ID. Priming queue from {seed_identifier}...")
        initial_match_ids = [seed_identifier]
        
    elif seed_identifier.isdigit():
        formatted_match_id = f"{PLATFORM.upper()}_{seed_identifier}"
        print(f"Seed recognized as a raw Game ID. Formatted to {formatted_match_id}. Priming queue...")
        initial_match_ids = [formatted_match_id]
        
    else:
        print(f"Seed recognized as a PUUID. Starting queue with this player...")
        puuids_to_explore.append(seed_identifier)
        initial_match_ids = []

    # If we started with a Match ID, process it right now to extract the first 10 players
    if initial_match_ids:
        match_url = f"https://{REGION}.api.riotgames.com/lol/match/v5/matches/{initial_match_ids[0]}"
        match_data = make_api_request(match_url)
        time.sleep(1.2) # Rate limit safety
        
        if match_data and 'metadata' in match_data:
            participants = match_data['metadata']['participants']
            puuids_to_explore.extend(participants) # Dump all 10 players into our queue!
        else:
            print("Failed to load the seed match. Aborting.")
            return



    # ==========================================
    # LOAD KNOWN PLAYERS INTO MEMORY
    # ==========================================
    known_puuids = set()
    if os.path.exists(MASTER_PLAYER_FILE):
        print("Loading known players from Master File into memory...")
        master_df = pd.read_csv(MASTER_PLAYER_FILE)
        if not master_df.empty and 'puuid' in master_df.columns:
            # We slice the first 15 chars just like your previous logic to match
            known_puuids = set(master_df['puuid'].astype(str))
            print(f"Loaded {len(known_puuids)} players. Ready to snowball!")
    
    print(f"\nStarting continuous snowball. Target: {target_new_players} new players.")

    # ==========================================
    # 2. THE CONTINUOUS LOOP
    # ==========================================
    while puuids_to_explore and saved_count < target_new_players:
        # Grab the first person in line
        current_puuid = puuids_to_explore.pop(0) 
        
        if current_puuid in explored_puuids:
            continue # Skip them if we already looked at their match history
            
        explored_puuids.add(current_puuid)
        print(f"\n--- Exploring PUUID: {current_puuid[:15]}... ({len(puuids_to_explore)} in queue) ---")
        
        # Pull their recent matches
        matches_url = f"https://{REGION}.api.riotgames.com/lol/match/v5/matches/by-puuid/{current_puuid}/ids?queue=420&start=0&count={matches_per_player}"
        match_ids = make_api_request(matches_url)
        time.sleep(1.2) # Rate limit safety
        
        if not match_ids:
            continue
            
        # Look inside those matches for new people
        for match_id in match_ids:
            if saved_count >= target_new_players:
                break # Stop instantly if we hit our global goal!
                
            match_url = f"https://{REGION}.api.riotgames.com/lol/match/v5/matches/{match_id}"
            match_data = make_api_request(match_url)
            
            if match_data and 'metadata' in match_data:
                participants = match_data['metadata']['participants']
                
                for p_puuid in participants:
                    if saved_count >= target_new_players:
                        break # Stop instantly
                        
                    # NEW: Instantly check our memory set instead of the hard drive         
                    if p_puuid not in known_puuids:
                        print(f"  -> Found new player! Fetching rank... ({saved_count + 1}/{target_new_players})")
                        rank = get_player_rank(p_puuid)
                        
                        # Save them to the master CSV
                        save_player_metadata(p_puuid, rank)
                        
                        # Add them to our memory set so we don't accidentally pull them again 5 seconds later
                        known_puuids.add(p_puuid)
                        
                        saved_count += 1
                        
                        # Add to queue
                        if p_puuid not in explored_puuids and p_puuid not in puuids_to_explore:
                            puuids_to_explore.append(p_puuid)
                            
    print(f"\nSnowball complete! Successfully gathered {saved_count} new players.")

## Player Data Gathering

In [150]:
print("Starting player discovery process...")
snowball_continuous(seed_identifier="7821896853", target_new_players=5000, matches_per_player=3)

Starting player discovery process...
Seed recognized as a raw Game ID. Formatted to EUW1_7821896853. Priming queue...
Loading known players from Master File into memory...
Loaded 5000 players. Ready to snowball!

Starting continuous snowball. Target: 5000 new players.

--- Exploring PUUID: ZibiEjskhRLyhHa... (9 in queue) ---

--- Exploring PUUID: yjQoykvS7t5yyEI... (8 in queue) ---
  -> Found new player! Fetching rank... (1/5000)
  -> Found new player! Fetching rank... (2/5000)
  -> Found new player! Fetching rank... (3/5000)
  -> Found new player! Fetching rank... (4/5000)
  -> Found new player! Fetching rank... (5/5000)
  -> Found new player! Fetching rank... (6/5000)
  -> Found new player! Fetching rank... (7/5000)
  -> Found new player! Fetching rank... (8/5000)
  -> Found new player! Fetching rank... (9/5000)

--- Exploring PUUID: gJhiH0m5XjHloOW... (16 in queue) ---

--- Exploring PUUID: usGvEmDTLSDAlPp... (15 in queue) ---

--- Exploring PUUID: QvyrrwwsZ4x1X77... (14 in queue) -

KeyboardInterrupt: 

## Match Data Gathering

In [161]:
# ==========================================
# Api SETTINGS
# ==========================================
MASTER_PLAYER_FILE = "Data/player_master_list.csv"
TARGET_PLAYERS_THIS_RUN = 50  # How many NEW players to scrape if no players have been scraped yet
MATCHES_PER_PLAYER = 300 # Target number of matches per player

print(f"Loading target players from {MASTER_PLAYER_FILE}...")

# If the master file doesn't exist, call the snowball function to create it!
if not os.path.exists(MASTER_PLAYER_FILE):
    print("Master player file not found! Calling snowball method to find players...")
    # Assuming you ran the cell defining this function previously:
    snowball_continuous("YOUR_SEED_PUUID_OR_MATCH_ID", target_new_players=50)

# Once we are sure the file exists, proceed with the Match Gathering
if os.path.exists(MASTER_PLAYER_FILE):
    # 1. Load the single master file into memory
    master_df = pd.read_csv(MASTER_PLAYER_FILE)
    
    # Drop accidental duplicates just to be safe
    master_df = master_df.drop_duplicates(subset=['puuid'])
    
    players_to_process = []
    skipped_count = 0
    
    print("Filtering out players who already have match data...")
    
    # 2. Extract PUUID and Rank, and do the Pre-Filter Check
    for index, row in master_df.iterrows():
        puuid = str(row['puuid'])
        rank = str(row['rank']) if 'rank' in row else "UNKNOWN"
        
        # Use [:15] to safely match the file names we create in save_player_data
        expected_filename = f"player_{puuid}_matches.csv"
        expected_filepath = os.path.join("Data/match/", expected_filename)
        
        if os.path.exists(expected_filepath):
            skipped_count += 1
        else:
            players_to_process.append({"puuid": puuid, "rank": rank})

    print(f"Skipped {skipped_count} players (already downloaded).")
    
    # 3. Enforce the Batch Limit
    if len(players_to_process) > TARGET_PLAYERS_THIS_RUN:
        players_to_process = players_to_process[:TARGET_PLAYERS_THIS_RUN]
        
    total_players = len(players_to_process)
    
    if total_players == 0:
        print("No new players left to scrape in the master list! Run your snowball function again to find more.")
    else:
        print(f"Found players needing data. Limiting this batch to {total_players} players.")

        # --- Start a timer for the entire script ---
        script_start_time = time.time()

        for player_index, player_dict in enumerate(players_to_process):
            target_player = player_dict['puuid']
            current_rank = player_dict['rank']
            
            # --- Calculate Global ETA based on fully completed players ---
            if player_index > 0:
                script_elapsed = time.time() - script_start_time
                avg_time_per_player = script_elapsed / player_index
                players_left = total_players - player_index
                
                global_eta_seconds = avg_time_per_player * players_left 
                
                g_mins, g_secs = divmod(int(global_eta_seconds), 60)
                g_hours, g_mins = divmod(g_mins, 60)
                global_eta_display = f" | Script ETA: {g_hours}h {g_mins}m {g_secs}s"
            else:
                global_eta_display = " | Script ETA: Calculating..."
                
            print(f"\n--- Processing Player {player_index + 1}/{total_players} (Rank: {current_rank}){global_eta_display} ---")
            
            # 1. Get their last Ranked games using our setting variable
            recent_match_ids = get_player_match_history(target_player, total_matches=MATCHES_PER_PLAYER)
            
            # 2. Parse the details for each game and store in a list
            all_match_records = []
            
            if recent_match_ids:
                total_matches = len(recent_match_ids)
                print(f"Parsing details for {total_matches} matches...")
                player_start_time = time.time()

                for index, m_id in enumerate(recent_match_ids):
                    if index > 0:
                        elapsed_time = time.time() - player_start_time
                        avg_time_per_match = elapsed_time / index
                        matches_left = total_matches - index
                        mins, secs = divmod(int(avg_time_per_match * matches_left), 60)
                        eta_display = f" | Local ETA: {mins}m {secs}s"
                    else:
                        eta_display = " | Local ETA: Calculating..."

                    print(f" -> Match {index + 1}/{total_matches} (ID: {m_id}){eta_display} | {global_eta_display}")
                    
                    record = parse_match_data(m_id, target_player)
                    
                    if record:
                        record['rank'] = current_rank # Inject the rank we loaded from their file!
                        all_match_records.append(record)
                    
                    time.sleep(1.2)
                    
                # 3. Convert and Save
                df = pd.DataFrame(all_match_records)
                if not df.empty and 'game_end_timestamp' in df.columns:
                    df['game_end_datetime'] = pd.to_datetime(df['game_end_timestamp'], unit='ms')
                
                save_player_data(df, target_player)

Loading target players from Data/player_master_list.csv...
Filtering out players who already have match data...
Skipped 10 players (already downloaded).
Found players needing data. Limiting this batch to 50 players.

--- Processing Player 1/50 (Rank: SILVER III) | Script ETA: Calculating... ---
Fetching up to 300 match IDs for player...
 -> Requesting 100 matches starting at index 0...
 -> Requesting 100 matches starting at index 100...
 -> Requesting 100 matches starting at index 200...
Successfully grabbed 300 total match IDs.
Parsing details for 300 matches...
 -> Match 1/300 (ID: EUW1_7822266242) | Local ETA: Calculating... |  | Script ETA: Calculating...
 -> Match 2/300 (ID: EUW1_7822235678) | Local ETA: 7m 24s |  | Script ETA: Calculating...
 -> Match 3/300 (ID: EUW1_7821915671) | Local ETA: 7m 8s |  | Script ETA: Calculating...
 -> Match 4/300 (ID: EUW1_7821863286) | Local ETA: 7m 2s |  | Script ETA: Calculating...
 -> Match 5/300 (ID: EUW1_7821807279) | Local ETA: 7m 0s |  | Sc

ChunkedEncodingError: Response ended prematurely